In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RESULTS_DIR = Path("evals/experiments")
PLOTS_DIR   = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)   # create if missing, no error if already exists

K_VALUES = [3, 4, 5, 6]
METRICS  = [
    "answer_correctness",
    "faithfulness",
    "response_relevancy",
    "context_precision",
    "context_recall",
]
INDEX_LABELS = {
    "sentence":              "Sentence (sliding-window) 512",
    "markdown":              "Markdown",
    "markdown_and_sentence": "Hybrid (Markdown + Sentence) 512",
}
INDEX_COLORS = {
    "sentence":              "#648FFF",   # blue
    "markdown":              "#FE6100",   # orange
    "markdown_and_sentence": "#785EF0",   # purple
}
INDEX_TYPES = list(INDEX_LABELS.keys())
# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------
def build_filename(index_type: str, k: int) -> Path:
    if index_type == "sentence":
        name = f"from_index_sentence_512_k_{k}_results.csv"
    elif index_type == "markdown":
        name = f"from_index_markdown_k_{k}_results.csv"
    else:
        name = f"from_index_markdown_and_sentence_512_k_{k}_results.csv"
    return RESULTS_DIR / name


def load_all_results() -> pd.DataFrame:
    frames = []
    for index_type in INDEX_TYPES:
        for k in K_VALUES:
            path = build_filename(index_type, k)
            if not path.exists():
                print(f"[WARNING] File not found, skipping: {path}")
                continue
            df = pd.read_csv(path).dropna(subset=["question"])
            df["index_type"] = index_type
            df["k"]          = k
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No CSVs found in '{RESULTS_DIR}'.")
    return pd.concat(frames, ignore_index=True)


def pass_stats(df: pd.DataFrame) -> pd.DataFrame:
    grp   = df.groupby(["index_type", "k"])
    stats = grp["judge_result"].apply(
        lambda s: (s.str.lower() == "pass").sum()
    ).rename("pass_count").reset_index()
    stats["total"]     = grp["judge_result"].count().values
    stats["pass_rate"] = stats["pass_count"] / stats["total"] * 100
    return stats


def metric_means(df: pd.DataFrame) -> pd.DataFrame:
    available = [m for m in METRICS if m in df.columns]
    # exclude rows where ground truth is absent (metrics are meaningless)
    df_filtered = df[df["answer_source"] != "absent for choice"]
    return df_filtered.groupby(["index_type", "k"])[available].mean().reset_index()

df    = load_all_results()
stats = pass_stats(df)
means = metric_means(df)

display(stats)
display(means.round(4))


# ---------------------------------------------------------------------------
# Plot 1 — Pass rate: grouped bar chart
# ---------------------------------------------------------------------------
def plot_pass_rate(stats: pd.DataFrame, save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.suptitle("Judge Pass Results by Index Type and k", fontsize=14, fontweight="bold")

    bar_width = 0.25
    x = np.arange(len(K_VALUES))

    for i, idx_type in enumerate(INDEX_TYPES):
        subset = stats[stats["index_type"] == idx_type].sort_values("k")
        values = subset["pass_rate"].values
        bars = ax.bar(
            x + i * bar_width, values,
            width=bar_width,
            label=INDEX_LABELS[idx_type],
            color=INDEX_COLORS[idx_type],
            edgecolor="white", linewidth=0.6,
        )
        for bar, v in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{v:.0f}%",
                ha="center", va="bottom", fontsize=7.5,
            )

    ax.set_xticks(x + bar_width)
    ax.set_xticklabels([f"k={k}" for k in K_VALUES])
    ax.set_ylabel("Pass rate (%)")
    ax.set_ylim(0, 110)
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_pass_rate(stats, PLOTS_DIR / "1_pass_rate.png")


# ---------------------------------------------------------------------------
# Plot 2 — RAGAS metrics: line chart per metric
# ---------------------------------------------------------------------------
def plot_metrics_lines(means: pd.DataFrame, save_path: Path) -> None:
    available = [m for m in METRICS if m in means.columns]
    ncols = 3
    nrows = (len(available) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), squeeze=False)
    fig.suptitle("RAGAS Metrics by Index Type and k", fontsize=14, fontweight="bold")

    for ax_idx, metric in enumerate(available):
        ax = axes[ax_idx // ncols][ax_idx % ncols]
        for idx_type in INDEX_TYPES:
            subset = means[means["index_type"] == idx_type].sort_values("k")
            ax.plot(
                subset["k"], subset[metric],
                marker="o", label=INDEX_LABELS[idx_type],
                color=INDEX_COLORS[idx_type], linewidth=2, markersize=6,
            )
        ax.set_title(metric.replace("_", " ").title(), fontsize=11)
        ax.set_xlabel("k")
        ax.set_ylabel("Score")
        ax.set_xticks(K_VALUES)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=7.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(linestyle="--", alpha=0.4)

    for ax_idx in range(len(available), nrows * ncols):
        axes[ax_idx // ncols][ax_idx % ncols].set_visible(False)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_metrics_lines(means, PLOTS_DIR / "2_metrics_lines.png")

# plot_heatmap(means, stats, PLOTS_DIR / "3_heatmap.png")
# ---------------------------------------------------------------------------
# Plot 3 — Heatmap: metric x configuration
# ---------------------------------------------------------------------------
def plot_heatmap(means: pd.DataFrame, stats: pd.DataFrame, save_path: Path) -> None:
    available = [m for m in METRICS if m in means.columns]

    configs, pass_row = [], []
    metric_matrix = {m: [] for m in available}

    for idx_type in INDEX_TYPES:
        for k in K_VALUES:
            configs.append(f"{INDEX_LABELS[idx_type]}\nk={k}")
            row_m = means[(means["index_type"] == idx_type) & (means["k"] == k)]
            row_s = stats[(stats["index_type"] == idx_type) & (stats["k"] == k)]
            for m in available:
                metric_matrix[m].append(row_m[m].values[0] if len(row_m) else np.nan)
            pass_row.append(row_s["pass_rate"].values[0] / 100 if len(row_s) else np.nan)

    all_labels = [m.replace("_", " ").title() for m in available] + ["Pass Rate"]
    data = np.array([metric_matrix[m] for m in available] + [pass_row])

    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.85), len(all_labels) * 0.75 + 1.5))
    im = ax.imshow(data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

    for r in range(data.shape[0]):
        for c in range(data.shape[1]):
            if not np.isnan(data[r, c]):
                ax.text(c, r, f"{data[r, c]:.2f}", ha="center", va="center", fontsize=7.5)

    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels(configs, fontsize=7, rotation=30, ha="right")
    ax.set_yticks(range(len(all_labels)))
    ax.set_yticklabels(all_labels, fontsize=9)

    for sep in [4, 8]:   # vertical separator between index-type groups
        ax.axvline(sep - 0.5, color="white", linewidth=2)

    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="Score (0–1)")
    ax.set_title("Metric Heatmap: Index Type × k", fontsize=13, fontweight="bold", pad=10)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


# ---------------------------------------------------------------------------
# Configuration for 1024 chunk size comparison
# ---------------------------------------------------------------------------
INDEX_LABELS_1024 = {
    "sentence_512":         "Sentence 512",
    "sentence_1024":         "Sentence 1024",
}
INDEX_COLORS_1024 = {
    "sentence_512":         "#0072B2",   # blue
    # "markdown_and_sentence_1024": "#009E73",  # uncomment when ready
    "sentence_1024":              "#E69F00",   # orange (same as before, it's the same index)
}
INDEX_TYPES_1024 = list(INDEX_LABELS_1024.keys())


def build_filename_1024(index_type: str, k: int) -> Path:
    if index_type == "sentence_1024":
        name = f"from_index_sentence_1024_k_{k}_results.csv"
    elif index_type == "sentence_512":
        name = f"from_index_sentence_512_k_{k}_results.csv"
    else:  # markdown — reuse existing files
        name = f"from_index_markdown_k_{k}_results.csv"
    return RESULTS_DIR / name


# load data for 1024 group
frames_1024 = []
for index_type in INDEX_TYPES_1024:
    for k in K_VALUES:
        path = build_filename_1024(index_type, k)
        if not path.exists():
            print(f"[WARNING] File not found, skipping: {path}")
            continue
        df_tmp = pd.read_csv(path).dropna(subset=["question"])
        df_tmp["index_type"] = index_type
        df_tmp["k"]          = k
        frames_1024.append(df_tmp)

df_1024    = pd.concat(frames_1024, ignore_index=True)
stats_1024 = pass_stats(df_1024)
means_1024 = metric_means(df_1024)


# ---------------------------------------------------------------------------
# Plots for 1024 group — reuse same functions, swap INDEX_TYPES/LABELS/COLORS
# ---------------------------------------------------------------------------

# temporarily override globals so the plot functions pick up the 1024 config
INDEX_TYPES,  _bak_types  = INDEX_TYPES_1024,  INDEX_TYPES
INDEX_LABELS, _bak_labels = INDEX_LABELS_1024, INDEX_LABELS
INDEX_COLORS, _bak_colors = INDEX_COLORS_1024, INDEX_COLORS

plot_pass_rate(stats_1024,        PLOTS_DIR / "1b_pass_rate_1024.png")
plot_metrics_lines(means_1024,    PLOTS_DIR / "2b_metrics_lines_1024.png")
# plot_heatmap(means_1024, stats_1024, PLOTS_DIR / "3b_heatmap_1024.png")

# restore original globals
INDEX_TYPES, INDEX_LABELS, INDEX_COLORS = _bak_types, _bak_labels, _bak_colors

[WARNING] File not found, skipping: evals/experiments/from_index_sentence_big_512_k_3_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_sentence_big_512_k_4_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_sentence_big_512_k_5_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_sentence_big_512_k_6_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_markdown_big_k_3_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_markdown_big_k_4_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_markdown_big_k_5_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_markdown_big_k_6_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_markdown_and_sentence_big_512_k_3_results.csv
[WARNING] File not found, skipping: evals/experiments/from_index_markdown_and_sentence_big_512_k_4_results.csv
[WARNING] 

FileNotFoundError: No CSVs found in 'evals/experiments'.